In [ ]:
from xgboost.spark import SparkXGBClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# 1. KHỞI TẠO SPARKSESSION
spark = SparkSession.builder \
    .appName("Weather_Preprocessing_Jupyter") \
    .master("local[*]") \
    .config("spark.driver.memory", "28g") \
    .config("spark.executor.memory", "28g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "32") \
    .getOrCreate()
# 2. ĐỌC DỮ LIỆU ĐÃ TIỀN XỬ LÝ
train_data = spark.read.parquet("train_data_folderkhongchuan.parquet")
test_data = spark.read.parquet("test_data_folderkhongchuan.parquet")
# 3. TẠO TẬP VALIDATION
print("⏳ Đang chuẩn bị tập Validation...")
# Chia tập train thành 80% để học thực sự và 20% để kiểm tra trong lúc học
train_sub, val_sub = train_data.randomSplit([0.8, 0.2], seed=42)
# Thêm cột 'is_val' để đánh dấu: False là Train, True là Validation
train_sub = train_sub.withColumn("is_val", F.lit(False))
val_sub = val_sub.withColumn("is_val", F.lit(True))
# Gộp lại thành một DataFrame tổng duy nhất để đưa vào mô hình
train_with_val = train_sub.union(val_sub)
# -------------------------------------------------------------------------
# 4. KHỞI TẠO VÀ HUẤN LUYỆN XGBOOST
xgb_classifier = SparkXGBClassifier(
    features_col="features",
    label_col="rain_label",
    num_workers=8,
    n_estimators=10000,
    max_depth=6,
    learning_rate=0.02,
    early_stopping_rounds=20,
    validation_indicator_col="is_val",
    scale_pos_weight=6
)
print("🚀 Đang tiến hành huấn luyện XGBoost (có Early Stopping)...")
xgb_model = xgb_classifier.fit(train_with_val)
print("✅ Huấn luyện thành công!")
# 5. DỰ ĐOÁN VÀ ĐÁNH GIÁ TRÊN TẬP TEST GỐC
print("🔮 Đang dự đoán trên tập Test...")
predictions = xgb_model.transform(test_data)
# 6. LƯU MÔ HÌNH
model_path = "xgboost_weather_model_kosmotekhongchuan"
xgb_model.write().overwrite().save(model_path)
print(f"💾 Đã lưu mô hình thành công tại thư mục: {model_path}")
# 7. ĐÁNH GIÁ ĐỘ CHÍNH XÁC CỦA MÔ HÌNH
evaluator = MulticlassClassificationEvaluator(labelCol="rain_label", predictionCol="prediction")
accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
precision_rain = evaluator.evaluate(predictions, {evaluator.metricName: "precisionByLabel", evaluator.metricLabel: 1.0})
recall_rain = evaluator.evaluate(predictions, {evaluator.metricName: "recallByLabel", evaluator.metricLabel: 1.0})
f1_rain = evaluator.evaluate(predictions, {evaluator.metricName: "fMeasureByLabel", evaluator.metricLabel: 1.0})

# Tạo nội dung báo cáo
report_content = f"""
==================================================
📊 BẢNG TỔNG HỢP CHỈ SỐ ĐÁNH GIÁ
==================================================
✔️ Accuracy  : {accuracy * 100:.2f}%
🎯 Precision : {precision_rain * 100:.2f}%
🎯 Recall    : {recall_rain * 100:.2f}%
🏆 F1-Score  : {f1_rain * 100:.2f}%
==================================================
"""

# In ra màn hình console như bình thường
print(report_content)

# Lưu nội dung vào file text
output_file = "evaluation_results_kosmotekhongchuan.txt"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(report_content)

print(f"📁 Đã xuất kết quả đánh giá thành công ra file: {output_file}")

# --- Hướng dẫn cách Load lại mô hình sau này ---
# from xgboost.spark import SparkXGBClassifierModel
# loaded_model = SparkXGBClassifierModel.load(model_path)
# predictions_new = loaded_model.transform(new_data)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/26 14:08:18 WARN Utils: Your hostname, admin2025RESNAN, resolves to a loopback address: 127.0.1.1; using 172.27.14.10 instead (on interface eth0)
26/03/26 14:08:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 14:08:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/26 14:08:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/26 14:08:22 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


⏳ Đang chuẩn bị tập Validation...
🚀 Đang tiến hành huấn luyện XGBoost (có Early Stopping)...


2026-03-26 14:08:37,253 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 8 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'learning_rate': 0.02, 'max_depth': 6, 'scale_pos_weight': 6, 'nthread': 1}
	train_call_kwargs_params: {'early_stopping_rounds': 20, 'verbose_eval': True, 'num_boost_round': 10000}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-03-26 14:08:58,658 INFO XGBoost-PySpark: _train_booster Training on CPUs 8]
[14:08:59] Task 0 got rank 0[14:08:59] Task 4 got rank 4

[14:08:59] Task 3 got rank 3
[14:08:59] Task 2 got rank 2
[14:08:59] Task 5 got rank 5
[14:08:59] Task 1 got rank 1[14:08:59] Task 7 got rank 7

[14:08:59] Task 6 got rank 6
[14:09:02] [0]	training-logloss:0.37282	validation-logloss:0.37320
[14:09:02] [1]	training-logloss:0.36769	validation-logloss:0.36809
[14:09:02] [2]	training-logloss:0.36277	validation-logloss:0.36318
[14:09:02] [3]	training-logloss:0.35802	validation-logloss:0.35844
[14:09:02] [4]	training-logloss:

✅ Huấn luyện thành công!
🔮 Đang dự đoán trên tập Test...


26/03/26 14:26:19 WARN TaskSetManager: Stage 16 contains a task of very large size (68343 KiB). The maximum recommended task size is 1000 KiB.


💾 Đã lưu mô hình thành công tại thư mục: xgboost_weather_model_kosmotekhongchuan


2026-03-26 14:26:23,382 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
2026-03-26 14:26:58,072 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
2026-03-26 14:27:30,796 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
2026-03-26 14:28:07,040 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs



📊 BẢNG TỔNG HỢP CHỈ SỐ ĐÁNH GIÁ (SAU KHI HYBRID RESAMPLING)
✔️ Accuracy  : 95.30%
🎯 Precision : 56.15%
🎯 Recall    : 68.98%
🏆 F1-Score  : 61.91%

📁 Đã xuất kết quả đánh giá thành công ra file: evaluation_results_kosmotekhongchuan.txt


In [ ]:
from xgboost.spark import SparkXGBClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# 1. Khởi tạo SparkSession
spark = SparkSession.builder \
    .appName("Weather_XGBoost_Final") \
    .master("local[*]") \
    .config("spark.driver.memory", "28g") \
    .config("spark.executor.memory", "28g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "32") \
    .getOrCreate()
# 2. ĐỌC DỮ LIỆU ĐÃ TIỀN XỬ LÝ
train_data = spark.read.parquet("train_data_folderkhongchuan.parquet")
test_data = spark.read.parquet("test_data_folderkhongchuan.parquet")
# 3. SMOTE
df_nang = train_data.filter(F.col("rain_label") == 0)
df_mua = train_data.filter(F.col("rain_label") == 1)
so_nang_cu = df_nang.count()
so_mua_cu = df_mua.count()
# Giảm 2 lần nắng
df_nang_giam = df_nang.sample(withReplacement=False, fraction=0.5, seed=42)
# Tăng 2 lần mưa
df_mua_tang = df_mua.sample(withReplacement=True, fraction=3.0, seed=42)
# Gộp 2 tập mới lại và xáo trộn (Shuffle)
train_hybrid = df_nang_giam.unionAll(df_mua_tang).orderBy(F.rand(seed=42))
# Đếm lại số lượng để tính tỷ lệ bập bênh mới
so_nang_moi = df_nang_giam.count()
so_mua_moi = df_mua_tang.count()
ty_le_lech_moi = so_nang_moi / so_mua_moi
print(f"📊 Tỷ lệ ban đầu -> Nắng: {so_nang_cu} | Mưa: {so_mua_cu}")
print(f"📊 Tỷ lệ lúc sau  -> Nắng: {so_nang_moi} | Mưa: {so_mua_moi}")
# 4. TẠO TẬP VALIDATION
print("⏳ Đang chuẩn bị tập Validation...")
train_sub, val_sub = train_hybrid.randomSplit([0.8, 0.2], seed=42)
train_sub = train_sub.withColumn("is_val", F.lit(False))
val_sub = val_sub.withColumn("is_val", F.lit(True))
train_with_val = train_sub.union(val_sub)
# 5. KHỞI TẠO VÀ HUẤN LUYỆN XGBOOST
xgb_classifier = SparkXGBClassifier(
    features_col="features",
    label_col="rain_label",
    num_workers=8,
    n_estimators=5000,
    max_depth=8,
    learning_rate=0.1,
    early_stopping_rounds=20,
    validation_indicator_col="is_val",
    scale_pos_weight=0.5
)

print("🚀 Đang tiến hành huấn luyện XGBoost (có Early Stopping)...")
xgb_model = xgb_classifier.fit(train_with_val)
print("✅ Huấn luyện thành công!")
# 6. DỰ ĐOÁN VÀ ĐÁNH GIÁ TRÊN TẬP TEST GỐC
print("🔮 Đang dự đoán trên tập Test...")
predictions = xgb_model.transform(test_data)
# 7. LƯU MÔ HÌNH
model_path = "xgboost_weather_model_cosmotekhongchuan"
xgb_model.write().overwrite().save(model_path)
print(f"💾 Đã lưu mô hình thành công tại thư mục: {model_path}")
# 8. ĐÁNH GIÁ ĐỘ CHÍNH XÁC CỦA MÔ HÌNH
evaluator = MulticlassClassificationEvaluator(labelCol="rain_label", predictionCol="prediction")
accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
precision_rain = evaluator.evaluate(predictions, {evaluator.metricName: "precisionByLabel", evaluator.metricLabel: 1.0})
recall_rain = evaluator.evaluate(predictions, {evaluator.metricName: "recallByLabel", evaluator.metricLabel: 1.0})
f1_rain = evaluator.evaluate(predictions, {evaluator.metricName: "fMeasureByLabel", evaluator.metricLabel: 1.0})

# Tạo nội dung báo cáo
report_content = f"""
==================================================
📊 BẢNG TỔNG HỢP CHỈ SỐ ĐÁNH GIÁ
==================================================
✔️ Accuracy  : {accuracy * 100:.2f}%
🎯 Precision : {precision_rain * 100:.2f}%
🎯 Recall    : {recall_rain * 100:.2f}%
🏆 F1-Score  : {f1_rain * 100:.2f}%
==================================================
"""

# In ra màn hình console như bình thường
print(report_content)

# Lưu nội dung vào file text
output_file = "evaluation_results_cosmotekhongchuan.txt"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(report_content)

print(f"📁 Đã xuất kết quả đánh giá thành công ra file: {output_file}")

26/03/26 14:28:39 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


📊 Tỷ lệ ban đầu -> Nắng: 813131 | Mưa: 47870
📊 Tỷ lệ lúc sau  -> Nắng: 407553 | Mưa: 143363
⏳ Đang chuẩn bị tập Validation...
🚀 Đang tiến hành huấn luyện XGBoost (có Early Stopping)...


2026-03-26 14:28:48,771 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 8 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 8, 'scale_pos_weight': 0.5, 'nthread': 1}
	train_call_kwargs_params: {'early_stopping_rounds': 20, 'verbose_eval': True, 'num_boost_round': 5000}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-03-26 14:28:54,706 INFO XGBoost-PySpark: _train_booster Training on CPUs 8]
[14:28:56] Task 2 got rank 2
[14:28:56] Task 1 got rank 1
[14:28:56] Task 0 got rank 0
[14:28:56] Task 4 got rank 4
[14:28:56] Task 5 got rank 5
[14:28:56] Task 6 got rank 6
[14:28:56] Task 7 got rank 7
[14:28:56] Task 3 got rank 3
[14:28:57] [0]	training-logloss:0.52752	validation-logloss:0.52834
[14:28:57] [1]	training-logloss:0.48786	validation-logloss:0.48897
[14:28:57] [2]	training-logloss:0.45759	validation-logloss:0.45895
[14:28:57] [3]	training-logloss:0.43374	validation-logloss:0.43543
[14:28:57] [4]	training-logloss:

✅ Huấn luyện thành công!
🔮 Đang dự đoán trên tập Test...


26/03/26 14:37:23 WARN TaskSetManager: Stage 65 contains a task of very large size (70957 KiB). The maximum recommended task size is 1000 KiB.


💾 Đã lưu mô hình thành công tại thư mục: xgboost_weather_model_cosmotekhongchuan


2026-03-26 14:37:24,668 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
2026-03-26 14:37:44,516 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
2026-03-26 14:38:05,899 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
2026-03-26 14:38:24,958 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs



📊 BẢNG TỔNG HỢP CHỈ SỐ ĐÁNH GIÁ (SAU KHI HYBRID RESAMPLING)
✔️ Accuracy  : 95.84%
🎯 Precision : 62.02%
🎯 Recall    : 64.02%
🏆 F1-Score  : 63.00%

📁 Đã xuất kết quả đánh giá thành công ra file: evaluation_results_cosmotekhongchuan.txt
